In [2]:
spark

In [3]:
from pyspark.sql import SparkSession
spark=SparkSession.builder\
.appName('SparkTable')\
.enableHiveSupport()\
.getOrCreate()

26/01/01 05:21:31 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [4]:
customer_schema='customer_id INT,name STRING ,city STRING,state STRING,country STRING ,registration_date STRING , is_active BOOLEAN'

In [5]:
customer__df=spark.read.format('csv').schema(customer_schema).load('/tmp/1MB/customers.csv')

In [8]:
#customer__df.write.format('csv').saveAsTable('default.customers_1mb')

In [7]:
spark.sql('describe extended customers_1mb').show(truncate=False)

+----------------------------+--------------------------------------------------------------+-------+
|col_name                    |data_type                                                     |comment|
+----------------------------+--------------------------------------------------------------+-------+
|customer_id                 |int                                                           |NULL   |
|name                        |string                                                        |NULL   |
|city                        |string                                                        |NULL   |
|state                       |string                                                        |NULL   |
|country                     |string                                                        |NULL   |
|registration_date           |string                                                        |NULL   |
|is_active                   |boolean                                             

In [9]:
! hadoop fs -ls -h  /user/hive/warehouse/customers_1mb*

Found 2 items
-rw-r--r--   2 root hadoop          0 2026-01-01 05:14 /user/hive/warehouse/customers_1mb/_SUCCESS
-rw-r--r--   2 root hadoop      1.0 M 2026-01-01 05:14 /user/hive/warehouse/customers_1mb/part-00000-415aaac1-ef76-478f-91b8-dc1301566295-c000.csv


In [10]:
spark.sql('select * from customers_1mb limit 5').show()

+-----------+----------+---------+-----------+-------+-----------------+---------+
|customer_id|      name|     city|      state|country|registration_date|is_active|
+-----------+----------+---------+-----------+-------+-----------------+---------+
|       NULL|      name|     city|      state|country|registration_date|     NULL|
|          0|Customer_0|     Pune|Maharashtra|  India|       2025-06-29|    false|
|          1|Customer_1|Bangalore| Tamil Nadu|  India|       2025-12-07|     true|
|          2|Customer_2|Hyderabad|    Gujarat|  India|       2025-10-27|     true|
|          3|Customer_3|Bangalore|  Karnataka|  India|       2025-10-17|    false|
+-----------+----------+---------+-----------+-------+-----------------+---------+



In [11]:
spark.sql('cache table default.customers_1mb') # Eager caching

DataFrame[]

In [12]:
spark.sql('select count(*) from default.customers_1mb').show()

+--------+
|count(1)|
+--------+
|   17654|
+--------+



In [13]:
spark.sql('select city,count(*) from default.customers_1mb group by city').show()

+---------+--------+
|     city|count(1)|
+---------+--------+
|    Delhi|    2200|
|  Kolkata|    2223|
|Hyderabad|    2242|
|     city|       1|
|Bangalore|    2211|
|Ahmedabad|    2198|
|  Chennai|    2194|
|   Mumbai|    2142|
|     Pune|    2243|
+---------+--------+



In [14]:
# AQE Read--> Adaptive Query Execution --> It makes the read Operations Faster

In [15]:
spark.sql('uncache table default.customers_1mb')

DataFrame[]

In [16]:
spark.sql('cache lazy table default.customers_1mb')

DataFrame[]

In [17]:
spark.sql('select count(*) from default.customers_1mb').show()

+--------+
|count(1)|
+--------+
|   17654|
+--------+



In [18]:
spark.sql('uncache table default.customers_1mb')

DataFrame[]

In [20]:
spark.catalog.clearCache()

In [21]:
spark.stop()